In [ ]:
# condition: in positive and negative patches, ONLY use the residue with SasaRate > 0.2
# Here extract all the testing patches

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages_CNN', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages_CNN')

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import random

import itertools
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'

In [ ]:
#@title Extract the testing patch information from the data frame, and then convert patch info into 15*20 matrix

# One-hot encoding for amino acid types
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

# One-hot encoding for secondary structure
def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

# One-hot encoding for SaSaRate (Now 5 levels)
def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

# Combine 3-d one-hot encoded vector for secondary structure with 5-d one-hot encoded vector for SaSaRate
# Resulting in a 15-d vector
def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

# NumPy is more efficient for vectorized operations
def calculate_distance(residue1, residue2):
    coord1 = np.array([residue1['SCx'], residue1['SCy'], residue1['SCz']])
    coord2 = np.array([residue2['SCx'], residue2['SCy'], residue2['SCz']])
    return np.linalg.norm(coord1 - coord2)

# Process each fold's training set
for fold in range(1, 6):

    # Specify the directory and filename of the combined data CSV
    dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data_test')
    file_name = 'all_data_with_SI.csv'

    save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4' + f'/fold_{fold}/data_test_processed_step1_with_SI/'
    os.makedirs(save_dir, exist_ok=True)

    dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4' + f'/fold_{fold}/data_test_processed_step1_with_SI/cnn_testing_patch_3points.txt'

    # Read the CSV file without setting the first column as an index
    combined_df = pd.read_csv(os.path.join(dir_name, file_name))
    filtered_combined_df = combined_df

    # Overwrite or create cnn_positive_patch_3points.txt at the beginning
    with open(save_dir + 'cnn_testing_patch_3points.txt', 'w') as f:
        f.write('')  # Writing an empty string to ensure the file is empty


    # Initialize an empty list to store all positive patches
    testing_patches = []

    # Set the maximum number of times a residue can be used
    max_residue_use = 3

    # Loop through each group (protein)
    for name, group in filtered_combined_df.groupby('name'):
        # take interacting and non-interacting residues
        all_residues = group[group['SaSaRate'] > 0.1]
        patches = []

        # Track residue usage count
        residue_count = {index: 0 for index in all_residues.index}

        for comb in itertools.combinations(all_residues.iterrows(), 3):
            index1, row1 = comb[0]
            index2, row2 = comb[1]
            index3, row3 = comb[2]

            # Skip this combination if any residue is used more than the allowed number of times
            if residue_count[index1] >= max_residue_use or residue_count[index2] >= max_residue_use or residue_count[index3] >= max_residue_use:
                continue

            d12 = calculate_distance(row1, row2)
            d23 = calculate_distance(row2, row3)
            d31 = calculate_distance(row3, row1)

            if d12 < 8.5 and d23 < 8.5 and d31 < 8.5:
                patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]
                patch_info = [[row1['AA'], row1['SS'], row1['SaSaRate']],
                              [row2['AA'], row2['SS'], row2['SaSaRate']],
                              [row3['AA'], row3['SS'], row3['SaSaRate']]]
                patches.append({'indices': patch_indices, 'info': patch_info})

                # Increment the usage count for these residues
                residue_count[index1] += 1
                residue_count[index2] += 1
                residue_count[index3] += 1

        # Output patches to a text file
        with open(dir, 'a') as f:
            f.write(f"Patches for protein {name}:\n")
            for patch in patches:
                f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

        print(f"Patches generated for protein {name}")

        # Convert each patch into a 15*20 matrix and append it to positive_patches
        for patch in patches:
            patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
            for aa_info in patch['info']:
                aa = aa_info[0]
                ss = aa_info[1]
                sasa = aa_info[2]

                aa_encoded = encode_aa(aa)
                ss_encoded = encode_ss(ss)
                sasa_encoded = encode_sasa(sasa)

                combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

                for i in range(20):
                    if aa_encoded[i] == 1:
                        patch_matrix[:, i] += combined_info  # Increment the value at the corresponding position

            testing_patches.append(patch_matrix)

    # Convert positive_patches to a NumPy array
    testing_patches_3points = np.array(testing_patches)

    # Save the NumPy array
    np.save(f'{save_dir}cnn_testing_patch_3points.npy', testing_patches_3points)

In [ ]:
import os
import numpy as np
import pandas as pd
import itertools
from tqdm import tqdm

# Set the maximum residue usage limit
max_residue_use = 3  # or adjust this based on your requirements

# Function definitions remain the same

# One-hot encoding for amino acid types
def encode_aa(aa):
    # Define amino acid types
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

def calculate_distance(residue1, residue2):
    coord1 = np.array([residue1['SCx'], residue1['SCy'], residue1['SCz']])
    coord2 = np.array([residue2['SCx'], residue2['SCy'], residue2['SCz']])
    return np.linalg.norm(coord1 - coord2)

# Process each fold's testing set
for fold in range(1, 6):
    # Specify the directory and filename of the combined data CSV
    dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data_test')
    file_name = 'all_data_with_SI.csv'

    save_dir = f'/content/drive/MyDrive/abdb_v4/fold_{fold}/data_test_processed_step1_with_SI/'
    os.makedirs(save_dir, exist_ok=True)

    output_file = save_dir + 'cnn_testing_patch_3points.npy'
    processed_proteins_file = save_dir + 'processed_proteins.txt'

    # Load previously saved testing patches if available
    testing_patches = []
    if os.path.exists(output_file):
        testing_patches = np.load(output_file).tolist()

    # Load the list of processed proteins to skip them if re-running
    processed_proteins = set()
    if os.path.exists(processed_proteins_file):
        with open(processed_proteins_file, 'r') as f:
            processed_proteins = set(line.strip() for line in f)

    # Read the CSV file without setting the first column as an index
    combined_df = pd.read_csv(os.path.join(dir_name, file_name))

    # Loop through each protein, skip if already processed
    for name, group in tqdm(combined_df.groupby('name'), desc=f'Processing fold {fold}'):
        if name in processed_proteins:
            continue

        # Take interacting and non-interacting residues
        all_residues = group[group['SaSaRate'] > 0.1]
        patches = []

        # Track residue usage count
        residue_count = {index: 0 for index in all_residues.index}

        for comb in itertools.combinations(all_residues.iterrows(), 3):
            index1, row1 = comb[0]
            index2, row2 = comb[1]
            index3, row3 = comb[2]

            if residue_count[index1] >= max_residue_use or residue_count[index2] >= max_residue_use or residue_count[index3] >= max_residue_use:
                continue

            d12 = calculate_distance(row1, row2)
            d23 = calculate_distance(row2, row3)
            d31 = calculate_distance(row3, row1)

            if d12 < 8.5 and d23 < 8.5 and d31 < 8.5:
                patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]
                patch_info = [[row1['AA'], row1['SS'], row1['SaSaRate']],
                              [row2['AA'], row2['SS'], row2['SaSaRate']],
                              [row3['AA'], row3['SS'], row3['SaSaRate']]]
                patches.append({'indices': patch_indices, 'info': patch_info})
                residue_count[index1] += 1
                residue_count[index2] += 1
                residue_count[index3] += 1

        # Convert each patch into a 15*20 matrix
        for patch in patches:
            patch_matrix = np.zeros((15, 20))
            for aa_info in patch['info']:
                aa, ss, sasa = aa_info
                aa_encoded = encode_aa(aa)
                ss_encoded = encode_ss(ss)
                sasa_encoded = encode_sasa(sasa)

                combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)

                for i in range(20):
                    if aa_encoded[i] == 1:
                        patch_matrix[:, i] += combined_info

            testing_patches.append(patch_matrix)

        # Append the protein name to processed_proteins to avoid reprocessing
        with open(processed_proteins_file, 'a') as f:
            f.write(f"{name}\n")

        # Save intermediate progress
        np.save(output_file, np.array(testing_patches))

    print(f"Completed fold {fold} and saved patches.")


In [ ]:
import numpy as np

# Load the patches from fold 4
output_file = '/content/drive/MyDrive/abdb_v4/fold_4/data_test_processed_step1_with_SI/cnn_testing_patch_3points.npy'
testing_patches = np.load(output_file)

# Print the content of the first patch
first_patch = testing_patches[0]
print("First patch (15x20 matrix):")
print(first_patch)

In [ ]:
import numpy as np

# Load the patches from fold 4
output_file = '/content/drive/MyDrive/abdb_v4/fold_4/data_test_processed_step1_with_SI/cnn_testing_patch_3points.npy'
testing_patches = np.load(output_file)

# Count the number of patches
num_patches = len(testing_patches)

# Count the number of unique proteins processed
processed_proteins_file = '/content/drive/MyDrive/abdb_v4/fold_4/data_test_processed_step1_with_SI/processed_proteins.txt'
with open(processed_proteins_file, 'r') as f:
    processed_proteins = set(line.strip() for line in f)

num_proteins = len(processed_proteins)

# Print the results
print(f"Number of patches: {num_patches}")
print(f"Number of proteins processed: {num_proteins}")


In [ ]:
# Load the patches from fold 4
output_file = '/content/drive/MyDrive/abdb_v4/fold_3/data_test_processed_step1_with_SI/cnn_testing_patch_3points.npy'
testing_patches = np.load(output_file)

# Count the number of patches
num_patches = len(testing_patches)

# Count the number of unique proteins processed
processed_proteins_file = '/content/drive/MyDrive/abdb_v4/fold_3/data_test_processed_step1_with_SI/processed_proteins.txt'
with open(processed_proteins_file, 'r') as f:
    processed_proteins = set(line.strip() for line in f)

num_proteins = len(processed_proteins)

# Print the results
print(f"Number of patches: {num_patches}")
print(f"Number of proteins processed: {num_proteins}")